# Feature Engineering

## Load Cleaned Data

In [2]:
import pandas as pd
import numpy as np
import os

results = pd.read_csv(os.path.join('..', 'Cleaned_Data', 'results_cleaned.csv'))
results['date'] = pd.to_datetime(results['date'])

print("Shape:", results.shape)
results.head()

Shape: (8068, 11)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,weight
0,2018-01-02,Iraq,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,True,0,1.0
1,2018-01-02,Oman,Bahrain,1.0,0.0,Gulf Cup,Kuwait City,Kuwait,True,1,1.0
2,2018-01-05,Oman,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,True,0,1.0
3,2018-01-07,Estonia,Sweden,1.0,1.0,Friendly,Abu Dhabi,United Arab Emirates,True,0,0.5
4,2018-01-11,Denmark,Sweden,0.0,1.0,Friendly,Abu Dhabi,United Arab Emirates,True,-1,0.5


## Rolling Form (Last 10 Games)

In [3]:
# For each team before each match, calculate win rate in last 10 games
# This tells the model how "in form" a team is right now

def get_team_form(df, team, date, n=10):
    team_matches = df[
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    ].tail(n)
    
    if len(team_matches) == 0:
        return 0.5  # neutral if no history
    
    wins = 0
    for _, row in team_matches.iterrows():
        if row['home_team'] == team and row['result'] == 1:
            wins += 1
        elif row['away_team'] == team and row['result'] == -1:
            wins += 1
    
    return wins / len(team_matches)

print("Form function defined!")

Form function defined!


## Average Goals Scored and Conceded

In [4]:
# For each team, calculate average goals scored and conceded in last 10 games
# This captures attacking and defensive strength

def get_avg_goals(df, team, date, n=10):
    team_matches = df[
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    ].tail(n)
    
    if len(team_matches) == 0:
        return 1.0, 1.0  # neutral defaults
    
    scored = []
    conceded = []
    
    for _, row in team_matches.iterrows():
        if row['home_team'] == team:
            scored.append(row['home_score'])
            conceded.append(row['away_score'])
        else:
            scored.append(row['away_score'])
            conceded.append(row['home_score'])
    
    return np.mean(scored), np.mean(conceded)

print("Goals function defined!")

Goals function defined!


## Build Feature Matrix

In [6]:
# Loops through every match and builds features for both teams
# This is the dataset our ML model will train on

features = []

for idx, row in results.iterrows():
    home = row['home_team']
    away = row['away_team']
    date = row['date']
    
    home_form = get_team_form(results, home, date)
    away_form = get_team_form(results, away, date)
    
    home_scored, home_conceded = get_avg_goals(results, home, date)
    away_scored, away_conceded = get_avg_goals(results, away, date)
    
    is_neutral = 1 if row['neutral'] else 0
    
    features.append({
        'date': date,
        'home_team': home,
        'away_team': away,
        'home_form': home_form,
        'away_form': away_form,
        'form_diff': home_form - away_form,
        'home_avg_scored': home_scored,
        'home_avg_conceded': home_conceded,
        'away_avg_scored': away_scored,
        'away_avg_conceded': away_conceded,
        'goal_diff': home_scored - away_scored,
        'is_neutral': is_neutral,
        'weight': row['weight'],
        'result': row['result']
    })

features_df = pd.DataFrame(features)
print("Feature matrix shape:", features_df.shape)
features_df.head()

Feature matrix shape: (8068, 14)


,date,home_team,away_team,home_form,away_form,form_diff,home_avg_scored,home_avg_conceded,away_avg_scored,away_avg_conceded,goal_diff,is_neutral,weight,result
0,2018-01-02,Iraq,United Arab Emirates,0.5,0.5,0.0,1.0,1.0,1.0,1.0,0.0,1,1.0,0
1,2018-01-02,Oman,Bahrain,0.5,0.5,0.0,1.0,1.0,1.0,1.0,0.0,1,1.0,1
2,2018-01-05,Oman,United Arab Emirates,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1,1.0,0
3,2018-01-07,Estonia,Sweden,0.5,0.5,0.0,1.0,1.0,1.0,1.0,0.0,1,0.5,0
4,2018-01-11,Denmark,Sweden,0.5,0.0,0.5,1.0,1.0,1.0,1.0,0.0,1,0.5,-1


## Save Feature Matrix

In [7]:
features_df.to_csv(os.path.join('..', 'Feature_Engineering', 'features.csv'), index=False)
print("Saved successfully!")

Saved successfully!
